# Data Retrieval — Bluesky

| Field | Details |
|---|---|
| **Time window** | 5 Jul 2024 – 4 Nov 2024 |
| **Source** | Bluesky Social — `atproto` Python client |
| **Method** | Paginated hashtag search via API — 62 keywords across three clusters |
| **Keywords** | 19 Trump/MAGA · 20 Harris/Democrats · 23 neutral (Biden, debates, voting) |
| **Volume** | 32,866 raw posts & replies (bronze) → **27,219** after cleaning (silver) |
| **Saved to** | `Data/1_Bronze/Bluesky/bsky_US_2024_raw.csv` · `Data/2_Silver/Bluesky/bsky_US_2024_posts.csv` |

### Bronze — bsky_US_2024_raw.csv · 32,866 rows × 14 columns

| Column | Description |
|---|---|
| `timestamp` | ISO 8601 post creation time (UTC) |
| `author` | Bluesky handle |
| `display` | Display name |
| `text` | Cleaned post text (URLs removed) |
| `candidate` | Buzz label: `TrumpBuzz` / `HarrisBuzz` / `ElectionBuzz` |
| `post_type` | `post` or `comment` |
| `likes` | Like count at collection time |
| `reposts` | Repost count |
| `replies` | Reply count |
| `mentions` | List of @-mentioned handles |
| `is_reply` | Boolean — is this a reply? |
| `parent_uri` | URI of parent post (if reply) |
| `query` | Hashtag query that retrieved this post |
| `uri` | Unique Bluesky post URI |

### Silver — bsky_US_2024_posts.csv · 27,219 rows × 15 columns

Same schema as bronze, plus `election` column; filtered to window, bots removed, quality-checked.


<!-- toc -->
## Contents
- [Bronze — bsky_US_2024_raw.csv · 32,866 rows × 14 columns](#bronze-bsky_us_2024_rawcsv-32866-rows-14-columns)
- [Silver — bsky_US_2024_posts.csv · 27,219 rows × 15 columns](#silver-bsky_us_2024_postscsv-27219-rows-15-columns)


In [2]:
import sys, os, re, time, warnings, ast
from collections import Counter

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from atproto import Client as BskyClient

warnings.filterwarnings("ignore")

In [3]:
import os, sys

config_path = os.path.join("..", "A. Lectures", "Lecture 1 Social network analysis")
sys.path.insert(0, config_path)
from bsky_config_students import BLUESKY_USERNAME, BLUESKY_APP_PASSWORD

client = BskyClient()
client.login(BLUESKY_USERNAME, BLUESKY_APP_PASSWORD)
print(f"Authenticated as: {BLUESKY_USERNAME}")

Authenticated as: vermeulen-anna.bsky.social


In [4]:
ELECTIONS = {
    "US_2024": {
        "since"        : "2024-07-05T00:00:00Z",
        "until"        : "2024-11-04T23:59:59Z",
        "election_date": "2024-11-05",
        "hashtags": [
            # ── Core election ──────────────────────────────────────────────
            "#Election2024", "#USElection2024", "#ElectionDay", "#Vote2024",
            "#Presidential2024", "#PresidentialElection", "#ElectionNight",
            "#AmericaDecides", "#Decision2024", "#VoterRegistration",
            "#EarlyVoting", "#MailInVoting", "#ElectionIntegrity",
            "#Battleground2024", "#SwingState",

            # ── Trump / Republican ─────────────────────────────────────────
            "#Trump2024", "#TrumpVance", "#VoteTrump", "#Trump",
            "#DonaldTrump", "#MAGA", "#MAGA2024", "#AmericaFirst",
            "#TrumpRally", "#Republicans", "#GOP", "#Republican",
            "#JDVance", "#Vance2024", "#VanceVP",
            "#RNC2024", "#RepublicanConvention",
            "#Project2025", "#TrumpDebate",

            # ── Harris / Democrat ──────────────────────────────────────────
            "#Harris2024", "#KamalaHarris2024", "#KamalaHarris",
            "#HarrisWalz", "#VoteHarris", "#VoteBlue", "#VoteKamala",
            "#Kamala2024", "#Kamala", "#Harris",
            "#TimWalz", "#Walz2024", "#WalzVP",
            "#DNC2024", "#DemConvention", "#DemocraticConvention",
            "#WeAreNotGoingBack", "#WinWithKamala",
            "#Democrats", "#Democrat",

            # ── Biden exit ─────────────────────────────────────────────────
            "#BidenDropsOut", "#BidenOut",
            "#BidenWithdraws",

            # ── Debates & key moments ──────────────────────────────────────
            "#PresidentialDebate", "#Debate2024", "#VPDebate",
            "#TrumpHarrisDebate", "#DebateNight",
        ],
    },
}

ACTIVE_ELECTION = "US_2024"
cfg             = ELECTIONS[ACTIVE_ELECTION]
MIN_TEXT_LENGTH = 30

print(f"Active election : {ACTIVE_ELECTION}")
print(f"Window          : {cfg['since'][:10]}  to  {cfg['until'][:10]}")
print(f"Hashtags        : {len(cfg['hashtags'])}")

Active election : US_2024
Window          : 2024-07-05  to  2024-11-04
Hashtags        : 62


In [5]:
# ── Text cleaning ─────────────────────────────────────────────────────────────
def clean_text(text):
    """Strip URLs and collapse whitespace."""
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ── Quality filters ────────────────────────────────────────────────────────────
NEWS_BOT_PATTERNS = [
    "forbes", "guardian", "bbc", "cnn", "reuters", "apnews",
    "skynews", "france24", "euronews", "washingtonpost", "nytimes",
    "independent", "trtworld", "straits_times", "abc", "news-feed",
    "theguardian", "huffpost", "politico", "axios",
]

def is_news_bot(handle):
    return any(p in handle.lower() for p in NEWS_BOT_PATTERNS)

def has_real_text(text):
    """Return True if text has enough non-hashtag/mention content."""
    clean = re.sub(r"#\w+|@\w+", "", text).strip()
    return len(clean) >= MIN_TEXT_LENGTH


# ── Buzz labelling — query-based ───────────────────────────────────────────────
# Labels indicate which hashtag cluster a post belongs to, NOT the author's stance.
# A post labelled "TrumpBuzz" was found via a Trump-related hashtag.
# It may be pro-Trump, anti-Trump, or simply reporting on Trump.

TRUMP_QUERIES = {q.lower() for q in [
    "#Trump2024", "#TrumpVance", "#VoteTrump", "#Trump", "#DonaldTrump",
    "#MAGA", "#MAGA2024", "#AmericaFirst", "#TrumpRally", "#Republicans",
    "#GOP", "#Republican", "#JDVance", "#Vance2024", "#VanceVP",
    "#RNC2024", "#RepublicanConvention", "#Project2025", "#TrumpDebate",
]}
HARRIS_QUERIES = {q.lower() for q in [
    "#Harris2024", "#KamalaHarris2024", "#KamalaHarris", "#HarrisWalz",
    "#VoteHarris", "#VoteBlue", "#VoteKamala", "#Kamala2024", "#Kamala",
    "#Harris", "#TimWalz", "#Walz2024", "#WalzVP", "#DNC2024",
    "#DemConvention", "#DemocraticConvention", "#WeAreNotGoingBack",
    "#WinWithKamala", "#Democrats", "#Democrat",
]}

def label_buzz(row):
    """
    Assign buzz label based on the hashtag query that retrieved the post.
    For comments (query='thread'), fall back to text-based keyword matching
    as a rough approximation — treat with caution.
    """
    query = str(row.get("query", "")).lower().strip()
    if query in TRUMP_QUERIES:
        return "TrumpBuzz"
    if query in HARRIS_QUERIES:
        return "HarrisBuzz"
    # Fallback for comments / neutral hashtags: keyword heuristic
    text = str(row.get("text", "")).lower()
    trump_kw  = ["trump", "maga", "donald", "republican", "gop", "vance"]
    harris_kw = ["harris", "kamala", "democrat", "democratic", "walz"]
    s_t = sum(k in text for k in trump_kw)
    s_h = sum(k in text for k in harris_kw)
    if s_t > s_h: return "TrumpBuzz"
    if s_h > s_t: return "HarrisBuzz"
    return "ElectionBuzz"


# ── Post search ────────────────────────────────────────────────────────────────
def search_posts(client, query, since=None, until=None):
    """Paginate through all posts matching a hashtag query."""
    collected, cursor = [], None
    while True:
        params = {"q": query, "limit": 100, "sort": "latest"}
        if cursor: params["cursor"] = cursor
        if since:  params["since"]  = since
        if until:  params["until"]  = until
        try:
            resp = client.app.bsky.feed.search_posts(params)
        except Exception as e:
            print(f"  API error: {e}"); break
        if not resp.posts: break
        for post in resp.posts:
            rec  = post.record
            raw  = getattr(rec, "text", "") or ""
            text = clean_text(raw)
            if not text:
                continue
            collected.append({
                "uri"        : post.uri,
                "author"     : post.author.handle,
                "display"    : getattr(post.author, "display_name", "") or "",
                "text"       : text,
                "timestamp"  : getattr(rec, "created_at", "") or "",
                "likes"      : post.like_count   or 0,
                "reposts"    : post.repost_count or 0,
                "replies"    : post.reply_count  or 0,
                "mentions"   : re.findall(r"@([\w.\-]+)", raw),
                "is_reply"   : bool(getattr(rec, "reply", None)),
                "post_type"  : "post",
                "parent_uri" : None,
                "query"      : query,
            })
        cursor = getattr(resp, "cursor", None)
        if not cursor: break
        time.sleep(0.5)
    return collected


# ── Comment fetcher ────────────────────────────────────────────────────────────
def fetch_comments(client, uri, depth=3):
    """
    Fetch all comments on a post recursively up to `depth` levels.
    Each comment stores the URI of its direct parent post/comment.
    """
    try:
        resp = client.app.bsky.feed.get_post_thread({"uri": uri, "depth": depth})
    except Exception as e:
        print(f"  Thread error: {e}")
        return []

    comments = []

    def walk(node, parent_uri=None):
        if node is None:
            return
        post = getattr(node, "post", None)
        if post is None:
            return
        rec         = getattr(post, "record", None)
        raw         = getattr(rec, "text", "") or ""
        text        = clean_text(raw)
        current_uri = post.uri
        if text and not is_news_bot(post.author.handle):
            comments.append({
                "uri"        : post.uri,
                "author"     : post.author.handle,
                "display"    : getattr(post.author, "display_name", "") or "",
                "text"       : text,
                "timestamp"  : getattr(rec, "created_at", "") or "",
                "likes"      : post.like_count   or 0,
                "reposts"    : post.repost_count or 0,
                "replies"    : post.reply_count  or 0,
                "mentions"   : re.findall(r"@([\w.\-]+)", raw),
                "is_reply"   : True,
                "post_type"  : "comment",
                "parent_uri" : parent_uri,
                "query"      : "thread",
            })
        for child in (getattr(node, "replies", None) or []):
            walk(child, parent_uri=current_uri)

    walk(getattr(resp, "thread", None))
    return comments


print("Helper functions ready: clean_text, is_news_bot, has_real_text, label_buzz, search_posts, fetch_comments")

Helper functions ready: clean_text, is_news_bot, has_real_text, label_buzz, search_posts, fetch_comments


In [6]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BRONZE_CSV = f"../Data/1_Bronze/Bluesky/bsky_{ACTIVE_ELECTION}_raw.csv"
SILVER_CSV = f"../Data/2_Silver/Bluesky/bsky_{ACTIVE_ELECTION}_posts.csv"


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(os.path.abspath(BRONZE_CSV)), exist_ok=True)
os.makedirs(os.path.dirname(os.path.abspath(SILVER_CSV)), exist_ok=True)

# ── Step 1: Scrape posts per hashtag ──────────────────────────────────────────
all_posts = []
hashtags  = cfg["hashtags"]

for i, tag in enumerate(hashtags, 1):
    print(f"[{i}/{len(hashtags)}] {tag} ...", end=" ", flush=True)
    posts = search_posts(client, tag, since=cfg["since"], until=cfg["until"])
    all_posts.extend(posts)
    print(f"{len(posts)} posts  (total: {len(all_posts):,})")

# Deduplicate before fetching comments
df_posts = pd.DataFrame(all_posts).drop_duplicates(subset="uri").reset_index(drop=True)
print(f"\n{len(df_posts):,} unique posts — fetching comments...\n")

# ── Step 2: Fetch comments for each post ──────────────────────────────────────
all_comments = []
for i, row in enumerate(df_posts.itertuples(), 1):
    if i % 100 == 0:
        print(f"  Comments: {i}/{len(df_posts)} posts processed  ({len(all_comments):,} comments so far)")
    all_comments.extend(fetch_comments(client, row.uri, depth=3))

print(f"\nComments collected: {len(all_comments):,}")

# ── Step 3: Merge, deduplicate and save to Bronze ─────────────────────────────
df_raw = pd.concat([df_posts, pd.DataFrame(all_comments)], ignore_index=True)
df_raw = df_raw.drop_duplicates(subset="uri").reset_index(drop=True)
df_raw.to_csv(BRONZE_CSV, index=False)

print(f"Bronze saved: {len(df_raw):,} unique rows → {BRONZE_CSV}")

In [7]:
# ── Cleaning: filter, label and save to Silver ────────────────────────────────
# NOTE: the 'candidate' column reflects *hashtag cluster* (buzz), not political stance.
#   TrumpBuzz   = found via a Trump-related hashtag  (may be pro- OR anti-Trump)
#   HarrisBuzz  = found via a Harris-related hashtag (may be pro- OR anti-Harris)
#   ElectionBuzz = found via a neutral election hashtag, or no clear match

if not os.path.exists(BRONZE_CSV):
    print("Bronze CSV not found — run the scraping cell first.")
else:
    df     = pd.read_csv(BRONZE_CSV)
    before = len(df)

    # Parse timestamps
    df["datetime"] = pd.to_datetime(df["timestamp"], format="ISO8601", utc=True)

    # Keep only posts within the election window
    df = df[
        (df["datetime"] >= cfg["since"]) &
        (df["datetime"] <= cfg["until"])
    ]

    # Remove news bots and posts with too little real text
    df = df[~df["author"].apply(is_news_bot)]
    df = df[df["text"].apply(has_real_text)]

    # Buzz label: which hashtag cluster was this post found in?
    df["candidate"] = df.apply(label_buzz, axis=1)

    # Sort chronologically
    df = df.sort_values("datetime").reset_index(drop=True)

    # Column order
    cols = ["timestamp", "author", "display", "text", "candidate", "post_type",
            "likes", "reposts", "replies", "mentions", "is_reply", "parent_uri",
            "query", "uri"]
    cols = [c for c in cols if c in df.columns]
    df   = df[cols]

    df.to_csv(SILVER_CSV, index=False)

    print(f"Filtered  : {before:,} → {len(df):,} rows  ({before - len(df):,} removed)")
    print(f"Date range: {df.timestamp.iloc[0][:10]} → {df.timestamp.iloc[-1][:10]}")
    print(f"Buzz labels: {df.candidate.value_counts().to_dict()}")
    print(f"Silver saved → {SILVER_CSV}")

Filtered  : 32,862 → 27,215 rows  (5,647 removed)
Date range: 2024-07-05 → 2024-11-04
Buzz labels: {'TrumpBuzz': 13780, 'HarrisBuzz': 6849, 'ElectionBuzz': 6586}
Silver saved → ../Data/2_Silver/Bluesky/bsky_US_2024_posts.csv
